# Préparation des données : consommation réalisée RTE

Objectif : transformer les mesures brutes de l'API (pas de 15 min, trous possibles) en une **série horaire continue**, exportée en CSV pour les notebooks d'évaluation puis le fine-tuning.

1. Récupérer l'historique brut (API RTE, par tranches)
2. Agréger au pas horaire et traiter les trous
3. Ajouter les features calendaires (pour le futur fine-tuning)
4. Exporter en CSV (index UTC)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from datetime import datetime

import holidays
import pandas as pd

from src.forecasting.dataset import enforce_hourly_continuity, fetch_realized_history

## 1. Historique brut

In [ ]:
ML_ENGINE_BASE_URL = os.environ.get("ML_ENGINE_BASE_URL", "http://localhost:8000")

# Plusieurs mois d'historique, nécessaires pour un backtest honnête (et le fine-tuning).
# L'API RTE limite la plage par appel, la période est donc découpée en tranches de 14 jours.
START = datetime(2024, 1, 1)
END = datetime(2026, 8, 1)

df = fetch_realized_history(ML_ENGINE_BASE_URL, START, END, chunk_days=14)
print(f"{len(df)} points bruts, {df.index.min()} -> {df.index.max()}")
df.head()

In [ ]:
EXPECTED_POINTS_PER_HOUR = 4  # RTE publie un point tous les quarts d'heure

# resample = grouper par heure (les heures vides existent, à NaN) ; agg = calculer
# pour chaque heure sa valeur (mean) et son indicateur de fiabilité (count).
hourly = df["megawatts"].resample("1h").agg(["mean", "count"])

# Heures partielles : `mean` a l'air normale mais repose sur trop peu de points —
# invisibles sans `count`. On mesure le phénomène avant de décider d'un traitement.
partial = hourly[hourly["count"].between(1, EXPECTED_POINTS_PER_HOUR - 1)]
print(f"{len(partial)} heures agrégées sur moins de {EXPECTED_POINTS_PER_HOUR} points")
# Remède à activer seulement si ce nombre devient significatif (seuil fin : une moyenne
# sur 3 vrais points reste plus fiable qu'une interpolation) :
# hourly.loc[hourly["count"] <= 2, "mean"] = float("nan")

megawatts = hourly["mean"].rename("megawatts")
print(f"{int(megawatts.isna().sum())} heures absentes sur {len(megawatts)}")

# Contrat « 1 ligne = 1 heure » : trous <= 6 h interpolés sur place (les timestamps ne
# bougent pas), trous plus longs refusés (ValueError) plutôt que de fausser fenêtres et lags.
df_resampled = enforce_hourly_continuity(megawatts, max_gap_hours=6).to_frame()
print(f"Série horaire continue : {len(df_resampled)} heures")

## 3. Features calendaires

Non utilisées par Chronos zero-shot (univarié : il ne lit que les mégawatts) — préparées pour le fine-tuning avec covariables. Calculées en **heure locale** : une pointe à 8 h à Paris doit être lue comme 8 h, pas 7 h UTC.

In [ ]:
paris_index = df_resampled.index.tz_convert("Europe/Paris")

df_resampled["day_of_week"] = paris_index.dayofweek
df_resampled["hour_of_day"] = paris_index.hour
df_resampled["month"] = paris_index.month
df_resampled["is_weekend"] = paris_index.dayofweek.isin([5, 6]).astype(int)

fr_bank_holidays = holidays.France(years=paris_index.year.unique().tolist())
df_resampled["is_bank_holiday"] = pd.Index(paris_index.date).isin(fr_bank_holidays).astype(int)

df_resampled.head()

## 4. Export

Index conservé en **UTC** : offsets homogènes dans le CSV (pas de mélange +01:00 / +02:00), relecture fiable.

In [ ]:
df_resampled.to_csv("../data/processed/energy_consumption_realized_resampled.csv", index=True)
print(f"Export : {len(df_resampled)} lignes horaires")